In [ ]:
!pip install lifelines shap lime captum scikit-survival --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle

# v3: log2-transformed expression gets its own folder, so the old raw-scale results are never loaded by mistake.
# The old folder is kept untouched and compared in E12.
RAW_CHECKPOINT_DIR = '/content/drive/MyDrive/pancreatic_paper_checkpoints'
CHECKPOINT_DIR     = '/content/drive/MyDrive/pancreatic_paper_checkpoints_log2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(name, obj):
    with open(f'{CHECKPOINT_DIR}/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f"Saved checkpoint: {name}")

def load_checkpoint(name):
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            print(f"Loaded checkpoint: {name}")
            return pickle.load(f)
    return None

Mounted at /content/drive


In [ ]:
import time
import json
from itertools import combinations

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler

from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

import shap
from lime.lime_tabular import LimeTabularExplainer
from captum.attr import IntegratedGradients
from scipy.stats import spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
df = pd.read_csv('tcga_paad_clean.csv')

stage_map = {
    'STAGE I': 1, 'STAGE IA': 1, 'STAGE IB': 1,
    'STAGE II': 2, 'STAGE IIA': 2, 'STAGE IIB': 2,
    'STAGE III': 3, 'STAGE IV': 4
}
df['STAGE_ENCODED'] = df['AJCC_PATHOLOGIC_TUMOR_STAGE'].map(stage_map).fillna(2)

gene_cols     = df.columns[5:1005].tolist()
clinical_cols = ['AGE', 'STAGE_ENCODED']

X_clinical = df[clinical_cols].values.astype(np.float32)
X_genomic  = df[gene_cols].values.astype(np.float32)
T          = df['OS_MONTHS'].values.astype(np.float32)
E          = df['OS_STATUS'].values.astype(np.float32)

# v3: the file holds raw RSEM values (median ~5,000, max ~3.7 million). Without a log transform, a few genes that are
# near zero in most tumours and enormous in a few (MUC2, SPINK4, CLCA1, GAST) dominate every attribution method.
if X_genomic.max() > 100:          # guard: never log-transform twice
    X_genomic = np.log2(X_genomic + 1)
print(f"Genomic values after log2: median {np.median(X_genomic):.2f}, max {X_genomic.max():.2f}")

# Clip genomic outliers before scaling
X_genomic = np.clip(X_genomic,
                    np.percentile(X_genomic, 1, axis=0),
                    np.percentile(X_genomic, 99, axis=0))

CLIN_DIM = X_clinical.shape[1]
feature_names = clinical_cols + gene_cols

print(f"Clinical features: {X_clinical.shape[1]}")
print(f"Genomic features:  {X_genomic.shape[1]}")
print(f"Patients:          {len(df)}")

Clinical features: 2
Genomic features:  1000
Patients:          177


In [ ]:
def cox_ph_loss(risk, T, E):
    order      = torch.argsort(T, descending=True)
    risk       = risk[order]
    E          = E[order]
    log_cumsum = torch.logcumsumexp(risk, dim=0)
    return -torch.mean((risk - log_cumsum) * E)

In [ ]:
class ClinicalMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


class GenomicMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1000, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


class GatedFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.clin_encoder = nn.Sequential(
            nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU()
        )
        self.gen_encoder = nn.Sequential(
            nn.Linear(1000, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4)
        )
        self.gate_clin = nn.Sequential(nn.Linear(192, 64), nn.Sigmoid())
        self.gate_gen  = nn.Sequential(nn.Linear(192, 128), nn.Sigmoid())
        self.head      = nn.Linear(192, 1)

    def forward(self, x_clin, x_gen):
        h_clin  = self.clin_encoder(x_clin)
        h_gen   = self.gen_encoder(x_gen)
        h       = torch.cat([h_clin, h_gen], dim=1)
        g_clin  = self.gate_clin(h)
        g_gen   = self.gate_gen(h)
        h_fused = torch.cat([h_clin * g_clin, h_gen * g_gen], dim=1)
        return self.head(h_fused).squeeze(1)

    def get_gates(self, x_clin, x_gen):
        h_clin = self.clin_encoder(x_clin)
        h_gen  = self.gen_encoder(x_gen)
        h      = torch.cat([h_clin, h_gen], dim=1)
        return self.gate_clin(h), self.gate_gen(h)

In [ ]:
def train_model(model, Xc, Xg, T, E, epochs=300, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    Xc = torch.tensor(Xc); Xg = torch.tensor(Xg)
    Tt = torch.tensor(T);  Et = torch.tensor(E)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        if isinstance(model, ClinicalMLP):
            risk = model(Xc)
        elif isinstance(model, GenomicMLP):
            risk = model(Xg)
        else:
            # any fusion model (GatedFusionModel, ConcatFusionModel, etc.) takes both inputs
            risk = model(Xc, Xg)
        loss = cox_ph_loss(risk, Tt, Et)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return model

In [ ]:
# Moved after the model definitions: unpickling fold models and rebuilding model_full
# need GatedFusionModel / train_model to exist first (it raised NameError in its old position).
cv_state = load_checkpoint('cv_state')
if cv_state:
    results_cv = cv_state['results_cv']
    fold_fusion_models = cv_state['fold_fusion_models']
    fold_scalers = cv_state['fold_scalers']
    fold_test_indices = cv_state['fold_test_indices']

model_state = load_checkpoint('model_full_state')
if model_state:
    model_full = GatedFusionModel()
    model_full.load_state_dict(model_state['model_full_state_dict'])
    model_full.eval()
    sc_full, sg_full = model_state['sc_full'], model_state['sg_full']
    X_all = model_state['X_all']

shap_state = load_checkpoint('shap_state')
if shap_state:
    shap_values = shap_state['shap_values']
    top20_genes = shap_state['top20_genes']

Loaded checkpoint: cv_state
Loaded checkpoint: model_full_state
Loaded checkpoint: shap_state


In [ ]:
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_cv = {'clinical': [], 'genomic': [], 'fusion': []}

fold_fusion_models = []
fold_scalers       = []
fold_test_indices  = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_clinical, E)):
    sc = RobustScaler(); sg = RobustScaler()
    Xc_tr = sc.fit_transform(X_clinical[train_idx]); Xc_te = sc.transform(X_clinical[test_idx])
    Xg_tr = sg.fit_transform(X_genomic[train_idx]);  Xg_te = sg.transform(X_genomic[test_idx])
    T_tr, E_tr = T[train_idx], E[train_idx]
    T_te, E_te = T[test_idx],  E[test_idx]

    for name, model_cls in [('clinical', ClinicalMLP), ('genomic', GenomicMLP), ('fusion', GatedFusionModel)]:
        torch.manual_seed(42)
        model = model_cls()
        model = train_model(model, Xc_tr, Xg_tr, T_tr, E_tr)
        model.eval()
        with torch.no_grad():
            if name == 'clinical':
                risk = model(torch.tensor(Xc_te)).numpy()
            elif name == 'genomic':
                risk = model(torch.tensor(Xg_te)).numpy()
            else:
                risk = model(torch.tensor(Xc_te), torch.tensor(Xg_te)).numpy()
        ci = concordance_index(T_te, -risk, E_te)
        results_cv[name].append(ci)
        print(f"Fold {fold+1} | {name:10s} C-index: {ci:.4f}")

        if name == 'fusion':
            fold_fusion_models.append(model)
            fold_scalers.append((sc, sg))
            fold_test_indices.append(test_idx)

print("\n--- Final CV Results (report these in your paper's Table 1) ---")
for name, scores in results_cv.items():
    print(f"{name:10s}: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

save_checkpoint('cv_state', {
    'results_cv': results_cv,
    'fold_fusion_models': fold_fusion_models,
    'fold_scalers': fold_scalers,
    'fold_test_indices': fold_test_indices,
})

Fold 1 | clinical   C-index: 0.5375
Fold 1 | genomic    C-index: 0.5736
Fold 1 | fusion     C-index: 0.5633
Fold 2 | clinical   C-index: 0.4893
Fold 2 | genomic    C-index: 0.4973
Fold 2 | fusion     C-index: 0.5214
Fold 3 | clinical   C-index: 0.6087
Fold 3 | genomic    C-index: 0.5924
Fold 3 | fusion     C-index: 0.6440
Fold 4 | clinical   C-index: 0.6400
Fold 4 | genomic    C-index: 0.6000
Fold 4 | fusion     C-index: 0.5400
Fold 5 | clinical   C-index: 0.7012
Fold 5 | genomic    C-index: 0.7027
Fold 5 | fusion     C-index: 0.6817

--- Final CV Results (report these in your paper's Table 1) ---
clinical  : 0.5953 +/- 0.0748
genomic   : 0.5932 +/- 0.0657
fusion    : 0.5901 +/- 0.0620
Saved checkpoint: cv_state


In [ ]:
class ConcatFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.clin_encoder = nn.Sequential(
            nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU()
        )
        self.gen_encoder = nn.Sequential(
            nn.Linear(1000, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4)
        )
        self.head = nn.Linear(192, 1)

    def forward(self, x_clin, x_gen):
        h_clin = self.clin_encoder(x_clin)
        h_gen  = self.gen_encoder(x_gen)
        h      = torch.cat([h_clin, h_gen], dim=1)
        return self.head(h).squeeze(1)


concat_scores = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X_clinical, E)):
    sc = RobustScaler(); sg = RobustScaler()
    Xc_tr = sc.fit_transform(X_clinical[train_idx]); Xc_te = sc.transform(X_clinical[test_idx])
    Xg_tr = sg.fit_transform(X_genomic[train_idx]);  Xg_te = sg.transform(X_genomic[test_idx])
    T_tr, E_tr = T[train_idx], E[train_idx]
    T_te, E_te = T[test_idx],  E[test_idx]

    torch.manual_seed(42)
    m = ConcatFusionModel()
    m = train_model(m, Xc_tr, Xg_tr, T_tr, E_tr)
    m.eval()
    with torch.no_grad():
        risk = m(torch.tensor(Xc_te), torch.tensor(Xg_te)).numpy()
    ci = concordance_index(T_te, -risk, E_te)
    concat_scores.append(ci)
    print(f"Fold {fold+1} | concat_fusion C-index: {ci:.4f}")

print(f"\nconcat_fusion: {np.mean(concat_scores):.4f} +/- {np.std(concat_scores):.4f}")

Fold 1 | concat_fusion C-index: 0.5711
Fold 2 | concat_fusion C-index: 0.5267
Fold 3 | concat_fusion C-index: 0.6304
Fold 4 | concat_fusion C-index: 0.5486
Fold 5 | concat_fusion C-index: 0.6877

concat_fusion: 0.5929 +/- 0.0587


In [ ]:
sc_full = RobustScaler(); sg_full = RobustScaler()
Xc_full = sc_full.fit_transform(X_clinical).astype(np.float32)
Xg_full = sg_full.fit_transform(X_genomic).astype(np.float32)

torch.manual_seed(42)
model_full = GatedFusionModel()
model_full = train_model(model_full, Xc_full, Xg_full, T, E, epochs=300)
model_full.eval()

X_all = np.hstack([Xc_full, Xg_full]).astype(np.float32)

def model_predict(X):
    X = X.astype(np.float32)
    Xc = torch.tensor(X[:, :CLIN_DIM])
    Xg = torch.tensor(X[:, CLIN_DIM:])
    with torch.no_grad():
        return model_full(Xc, Xg).numpy()

print("Final model trained. X_all shape:", X_all.shape)

save_checkpoint('model_full_state', {
    'model_full_state_dict': model_full.state_dict(),
    'sc_full': sc_full,
    'sg_full': sg_full,
    'X_all': X_all,
})

Final model trained. X_all shape: (177, 1002)
Saved checkpoint: model_full_state


In [ ]:
import shap
import time
t0 = time.time()
background   = shap.sample(X_all, 30)
explainer    = shap.KernelExplainer(model_predict, background)
shap_values  = explainer.shap_values(X_all, nsamples=200, l1_reg='num_features(50)')
shap_runtime = time.time() - t0
print(f"SHAP runtime: {shap_runtime:.1f}s")

shap.summary_plot(shap_values, X_all, feature_names=feature_names, max_display=20, show=False)
plt.savefig('shap_beeswarm.png', dpi=500, bbox_inches='tight')
plt.close()
print("Saved: shap_beeswarm.png")

shap_clin = np.abs(shap_values[:, :CLIN_DIM]).sum(axis=1)
shap_gen  = np.abs(shap_values[:, CLIN_DIM:]).sum(axis=1)
total     = shap_clin + shap_gen
clinical_pct = (shap_clin / total).mean() * 100
genomic_pct  = (shap_gen  / total).mean() * 100
print(f"\nClinical attribution: {clinical_pct:.1f}%")
print(f"Genomic attribution:  {genomic_pct:.1f}%")

plt.figure(figsize=(5, 5))
plt.bar(['Clinical', 'Genomic'], [clinical_pct, genomic_pct], color=['#4C72B0', '#DD8452'])
plt.ylabel('Mean SHAP Attribution (%)')
plt.title('Modality-Level SHAP Attribution\n(TCGA-PAAD)')
for i, v in enumerate([clinical_pct, genomic_pct]):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')
plt.savefig('modality_attribution.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: modality_attribution.png")
'''
gene_shap   = np.abs(shap_values[:, CLIN_DIM:]).mean(axis=0)
gene_shap_c = np.clip(gene_shap, 0, np.percentile(gene_shap, 99))
top20_idx   = np.argsort(gene_shap_c)[::-1][:20]
top20_genes = [(gene_cols[i], gene_shap_c[i]) for i in top20_idx]
print("\nTop 20 SHAP genes:")
for gene, val in top20_genes:
    print(f"  {gene}: {val:.4f}")
'''
gene_shap = np.abs(shap_values[:, CLIN_DIM:]).mean(axis=0)
top20_idx = np.argsort(-gene_shap)[:20]
top20_genes = [(gene_cols[i], gene_shap[i]) for i in top20_idx]

print("\nTop 20 SHAP genes (unclipped, true ranking):")
for gene, val in top20_genes:
    print(f"  {gene}: {val:.6f}")

  0%|          | 0/177 [00:00<?, ?it/s]

SHAP runtime: 42.3s


/tmp/ipykernel_1922/3780068786.py:10: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_all, feature_names=feature_names, max_display=20, show=False)


Saved: shap_beeswarm.png

Clinical attribution: 1.8%
Genomic attribution:  98.2%
Saved: modality_attribution.png

Top 20 SHAP genes (unclipped, true ranking):
  MUC2: 2.591218
  LOXL2: 2.075035
  KRT7: 1.896557
  BCAS1: 1.863048
  CLCA1: 1.179590
  PTMS: 1.047598
  DEFB1: 1.012197
  CTSK: 0.959827
  CCND1: 0.951762
  PDZK1IP1: 0.893465
  ITLN1: 0.885021
  REG3G: 0.741073
  SPINK4: 0.703156
  IGFBP3: 0.699645
  CAPG: 0.662861
  CNN2: 0.662195
  MIAT: 0.594466
  CFH: 0.588062
  CTSD: 0.578673
  ITPR3: 0.469805


In [ ]:
cosmic_paad = {
    'KRAS', 'TP53', 'SMAD4', 'CDKN2A', 'ARID1A',
    'ROBO2', 'KDM6A', 'RNF43', 'TGFBR2', 'BRAF',
    'GNAS', 'MYC', 'PTEN', 'RB1', 'STK11'
}

print("Top 20 SHAP genes -- COSMIC PDAC driver annotation:")
print(f"{'Rank':<6}{'Gene':<15}{'Mean |SHAP|':<15}{'COSMIC Driver'}")
print("-" * 55)
for rank, (gene, val) in enumerate(top20_genes, 1):
    is_driver = "YES" if gene in cosmic_paad else "-"
    print(f"{rank:<6}{gene:<15}{val:<15.4f}{is_driver}")

overlap = [g for g, _ in top20_genes if g in cosmic_paad]
print(f"\nOverlap with COSMIC PDAC drivers: {len(overlap)}/20 -> {overlap}")

Top 20 SHAP genes -- COSMIC PDAC driver annotation:
Rank  Gene           Mean |SHAP|    COSMIC Driver
-------------------------------------------------------
1     MUC2           2.5912         -
2     LOXL2          2.0750         -
3     KRT7           1.8966         -
4     BCAS1          1.8630         -
5     CLCA1          1.1796         -
6     PTMS           1.0476         -
7     DEFB1          1.0122         -
8     CTSK           0.9598         -
9     CCND1          0.9518         -
10    PDZK1IP1       0.8935         -
11    ITLN1          0.8850         -
12    REG3G          0.7411         -
13    SPINK4         0.7032         -
14    IGFBP3         0.6996         -
15    CAPG           0.6629         -
16    CNN2           0.6622         -
17    MIAT           0.5945         -
18    CFH            0.5881         -
19    CTSD           0.5787         -
20    ITPR3          0.4698         -

Overlap with COSMIC PDAC drivers: 0/20 -> []


In [ ]:
with torch.no_grad():
    all_risk = model_full(torch.tensor(Xc_full), torch.tensor(Xg_full)).numpy()

median_risk = np.median(all_risk)
high_risk   = all_risk >= median_risk
low_risk    = all_risk <  median_risk

results_lr = logrank_test(T[high_risk], T[low_risk], E[high_risk], E[low_risk])
print(f"Log-rank p-value: {results_lr.p_value:.6f}")

kmf_high = KaplanMeierFitter()
kmf_low  = KaplanMeierFitter()

plt.figure(figsize=(8, 6))
ax = plt.gca()
kmf_high.fit(T[high_risk], E[high_risk], label='High Risk')
kmf_high.plot_survival_function(ax=ax, color='red')
kmf_low.fit(T[low_risk], E[low_risk], label='Low Risk')
kmf_low.plot_survival_function(ax=ax, color='blue')
plt.title('Kaplan-Meier Survival Curves\nHigh vs Low Risk -- TCGA-PAAD')
plt.xlabel('Overall Survival (Months)')
plt.ylabel('Survival Probability')
plt.savefig('km_curve.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: km_curve.png")

save_checkpoint('shap_state', {'shap_values': shap_values, 'top20_genes': top20_genes})

Log-rank p-value: 0.000000
Saved: km_curve.png
Saved checkpoint: shap_state


In [ ]:
def get_gate_genomic_fraction(model, X_all_np):
    xc = torch.tensor(X_all_np[:, :CLIN_DIM].astype(np.float32))
    xg = torch.tensor(X_all_np[:, CLIN_DIM:].astype(np.float32))
    with torch.no_grad():
        g_clin, g_gen = model.get_gates(xc, xg)
    mg_clin = g_clin.mean().item()
    mg_gen  = g_gen.mean().item()
    return mg_gen / (mg_gen + mg_clin + 1e-10)


def make_numpy_predict_fn(model):
    def predict_fn(X):
        X = X.astype(np.float32)
        xc = torch.tensor(X[:, :CLIN_DIM])
        xg = torch.tensor(X[:, CLIN_DIM:])
        with torch.no_grad():
            return model(xc, xg).numpy()
    return predict_fn


class WrappedTorchModel(nn.Module):
    def __init__(self, base_model, clin_dim=CLIN_DIM):
        super().__init__()
        self.base = base_model
        self.clin_dim = clin_dim
    def forward(self, x):
        return self.base(x[:, :self.clin_dim], x[:, self.clin_dim:])


def compute_lime_global(model, X_all_np, feature_names, num_samples=300):
    t0 = time.time()
    predict_fn = make_numpy_predict_fn(model)
    explainer = LimeTabularExplainer(
        training_data=X_all_np, feature_names=feature_names,
        mode="regression", discretize_continuous=False,
    )
    n_features = X_all_np.shape[1]
    attributions = np.zeros_like(X_all_np, dtype=float)
    for i in range(X_all_np.shape[0]):
        exp = explainer.explain_instance(X_all_np[i], predict_fn, num_features=n_features, num_samples=num_samples)
        for feat_idx, weight in dict(exp.as_map()[1]).items():
            attributions[i, feat_idx] = weight
        if i % 30 == 0:
            print(f"    LIME: {i}/{X_all_np.shape[0]}")
    return np.mean(np.abs(attributions), axis=0), time.time() - t0


def compute_ig_global(model, X_all_np):
    t0 = time.time()
    wrapped = WrappedTorchModel(model)
    wrapped.eval()
    X_t = torch.tensor(X_all_np.astype(np.float32), requires_grad=True)
    baseline_vec = np.median(X_all_np, axis=0)
    baseline = torch.tensor(np.tile(baseline_vec, (X_all_np.shape[0], 1)).astype(np.float32))
    ig = IntegratedGradients(wrapped)
    attributions, _ = ig.attribute(X_t, baselines=baseline, n_steps=50, return_convergence_delta=True)
    return np.mean(np.abs(attributions.detach().numpy()), axis=0), time.time() - t0


def compute_pfi_global(model, X_all_np, T, E, n_repeats=10, random_state=42):
    t0 = time.time()
    rng = np.random.RandomState(random_state)
    predict_fn = make_numpy_predict_fn(model)
    baseline_preds = predict_fn(X_all_np)
    baseline_cindex = concordance_index(T, -baseline_preds, E)
    n_features = X_all_np.shape[1]
    importances = np.zeros(n_features)
    for f in range(n_features):
        drops = []
        for _ in range(n_repeats):
            X_perm = X_all_np.copy()
            X_perm[:, f] = rng.permutation(X_perm[:, f])
            perm_cindex = concordance_index(T, -predict_fn(X_perm), E)
            drops.append(baseline_cindex - perm_cindex)
        importances[f] = np.mean(drops)
        if f % 100 == 0:
            print(f"    PFI: {f}/{n_features}")
    return importances, time.time() - t0


def compute_pdp_global(model, X_all_np, candidate_idx, n_grid_points=10):
    t0 = time.time()
    predict_fn = make_numpy_predict_fn(model)
    full_vector = np.zeros(X_all_np.shape[1])
    for feat_idx in candidate_idx:
        grid = np.percentile(X_all_np[:, feat_idx], np.linspace(10, 90, n_grid_points))
        curve = []
        for val in grid:
            X_mod = X_all_np.copy()
            X_mod[:, feat_idx] = val
            curve.append(predict_fn(X_mod).mean())
        full_vector[feat_idx] = max(curve) - min(curve)
    return full_vector, time.time() - t0


def faithfulness_score(model, X_all_np, global_importance, T, E, top_k):
    predict_fn = make_numpy_predict_fn(model)
    baseline_values = np.median(X_all_np, axis=0)
    original_cindex = concordance_index(T, -predict_fn(X_all_np), E)
    top_k_idx = np.argsort(-global_importance)[:top_k]
    X_perturbed = X_all_np.copy()
    X_perturbed[:, top_k_idx] = baseline_values[top_k_idx]
    perturbed_cindex = concordance_index(T, -predict_fn(X_perturbed), E)
    return original_cindex - perturbed_cindex


def stability_score(per_fold_importances, top_k):
    top_sets = [set(np.argsort(-imp)[:top_k]) for imp in per_fold_importances]
    jaccards = []
    for a, b in combinations(top_sets, 2):
        inter, union = len(a & b), len(a | b)
        jaccards.append(inter / union if union > 0 else 0.0)
    return np.mean(jaccards), np.std(jaccards)


def correlation_aware_stability(X_genomic_full, per_fold_importances, top_k, clin_dim=CLIN_DIM, corr_threshold=0.8):
    n_genes = X_genomic_full.shape[1]
    corr_matrix = np.corrcoef(X_genomic_full.T)
    parent = list(range(n_genes))
    def find(x):
        while parent[x] != x: x = parent[x]
        return x
    def union(x, y):
        px, py = find(x), find(y)
        if px != py: parent[px] = py
    for i in range(n_genes):
        for j in range(i + 1, n_genes):
            if abs(corr_matrix[i, j]) > corr_threshold:
                union(i, j)
    feat_to_cluster = {clin_dim + i: find(i) for i in range(n_genes)}
    for i in range(clin_dim):
        feat_to_cluster[i] = f"clinical_{i}"
    cluster_top_sets = []
    for imp in per_fold_importances:
        top_k_idx = np.argsort(-imp)[:top_k]
        cluster_top_sets.append(set(feat_to_cluster[idx] for idx in top_k_idx))
    jaccards = []
    for a, b in combinations(cluster_top_sets, 2):
        inter, union_ = len(a & b), len(a | b)
        jaccards.append(inter / union_ if union_ > 0 else 0.0)
    return np.mean(jaccards), np.std(jaccards), len(set(feat_to_cluster.values()))


def gate_agreement_score(model, X_all_np, global_importance, clin_dim=CLIN_DIM):
    gate_genomic_fraction = get_gate_genomic_fraction(model, X_all_np)
    method_genomic_fraction = global_importance[clin_dim:].sum() / (global_importance.sum() + 1e-10)
    agreement = 1 - abs(gate_genomic_fraction - method_genomic_fraction)
    return agreement, gate_genomic_fraction, method_genomic_fraction

print("Comparison functions defined.")

Comparison functions defined.


In [ ]:
# --- Compute global importances for LIME, IG, PFI, PDP (SHAP already have) ---
shap_global = np.mean(np.abs(shap_values), axis=0)

print("Computing LIME...")
lime_global, lime_runtime = compute_lime_global(model_full, X_all, feature_names)

print("Computing Integrated Gradients...")
ig_global, ig_runtime = compute_ig_global(model_full, X_all)

print("Computing PFI...")
pfi_global, pfi_runtime = compute_pfi_global(model_full, X_all, T, E)

print("Computing PDP (top-25 candidates only)...")
candidates = set()
for imp in [shap_global, lime_global, ig_global, pfi_global]:
    candidates |= set(np.argsort(-imp)[:25])
pdp_global, pdp_runtime = compute_pdp_global(model_full, X_all, list(candidates))

runtimes = {
    'SHAP': shap_runtime, 'LIME': lime_runtime, 'IG': ig_runtime,
    'PFI': pfi_runtime, 'PDP': pdp_runtime,
}
global_importances = {
    'SHAP': shap_global, 'LIME': lime_global, 'IG': ig_global,
    'PFI': pfi_global, 'PDP': pdp_global,
}
print("\nRuntimes (s):", runtimes)

Computing LIME...
    LIME: 0/177
    LIME: 30/177
    LIME: 60/177
    LIME: 90/177
    LIME: 120/177
    LIME: 150/177
Computing Integrated Gradients...
Computing PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Computing PDP (top-25 candidates only)...

Runtimes (s): {'SHAP': 42.32993459701538, 'LIME': 35.70045733451843, 'IG': 0.5902588367462158, 'PFI': 68.74542951583862, 'PDP': 3.125603199005127}


In [ ]:
# --- Faithfulness ---
comparison_results = {}
for name, imp in global_importances.items():
    for k in [5, 10]:
        comparison_results[f"{name}_faithfulness_top{k}"] = faithfulness_score(
            model_full, X_all, imp, T, E, top_k=k
        )

# --- Gate agreement ---
model_gate_genomic_fraction = get_gate_genomic_fraction(model_full, X_all)
for name, imp in global_importances.items():
    agreement, gate_frac, method_frac = gate_agreement_score(model_full, X_all, imp)
    comparison_results[f"{name}_gate_agreement"] = agreement
    comparison_results[f"{name}_genomic_fraction"] = method_frac
comparison_results["model_gate_genomic_fraction"] = model_gate_genomic_fraction

print("Faithfulness + gate agreement done.")
for k, v in comparison_results.items():
    print(f"  {k}: {v}")

Faithfulness + gate agreement done.
  SHAP_faithfulness_top5: 0.03156866999349106
  SHAP_faithfulness_top10: 0.03156866999349106
  LIME_faithfulness_top5: 0.01714037752223918
  LIME_faithfulness_top10: 0.046105445866782446
  IG_faithfulness_top5: 0.06118463875027125
  IG_faithfulness_top10: 0.06530700802777178
  PFI_faithfulness_top5: 0.03601648947711
  PFI_faithfulness_top10: 0.05565198524625736
  PDP_faithfulness_top5: 0.045454545454545525
  PDP_faithfulness_top10: 0.05185506617487534
  SHAP_gate_agreement: 0.6073138747329776
  SHAP_genomic_fraction: 0.9459847782770674
  LIME_gate_agreement: 0.5565276908862428
  LIME_genomic_fraction: 0.9967709621238022
  IG_gate_agreement: 0.571242094039917
  IG_genomic_fraction: 0.9820565581321716
  PFI_gate_agreement: 0.59212910078623
  PFI_genomic_fraction: 0.961169552223815
  PDP_gate_agreement: 0.7218607544556497
  PDP_genomic_fraction: 0.8314378985543953
  model_gate_genomic_fraction: 0.553298653010045


In [ ]:
# --- Stability across folds (SLOW) ---
per_fold_lime, per_fold_ig, per_fold_pfi = [], [], []

for i, fold_model in enumerate(fold_fusion_models):
    test_idx = fold_test_indices[i]
    sc, sg = fold_scalers[i]
    Xc_te = sc.transform(X_clinical[test_idx])
    Xg_te = sg.transform(X_genomic[test_idx])
    X_fold = np.hstack([Xc_te, Xg_te]).astype(np.float32)
    T_fold, E_fold = T[test_idx], E[test_idx]

    print(f"Fold {i+1}/5 -- LIME...")
    lime_f, _ = compute_lime_global(fold_model, X_fold, feature_names, num_samples=150)
    per_fold_lime.append(lime_f)

    print(f"Fold {i+1}/5 -- IG...")
    ig_f, _ = compute_ig_global(fold_model, X_fold)
    per_fold_ig.append(ig_f)

    print(f"Fold {i+1}/5 -- PFI...")
    pfi_f, _ = compute_pfi_global(fold_model, X_fold, T_fold, E_fold, n_repeats=5)
    per_fold_pfi.append(pfi_f)

    print(f"Fold {i+1}/5 done.")

for name, per_fold in [('LIME', per_fold_lime), ('IG', per_fold_ig), ('PFI', per_fold_pfi)]:
    mean_j, std_j = stability_score(per_fold, top_k=10)
    comparison_results[f"{name}_stability_jaccard_mean"] = mean_j
    comparison_results[f"{name}_stability_jaccard_std"] = std_j
    mean_jc, std_jc, n_clust = correlation_aware_stability(X_genomic, per_fold, top_k=10)
    comparison_results[f"{name}_stability_corr_aware_mean"] = mean_jc
    comparison_results[f"{name}_stability_corr_aware_std"] = std_jc

print("\nStability scoring done.")

Fold 1/5 -- LIME...
    LIME: 0/36
    LIME: 30/36
Fold 1/5 -- IG...
Fold 1/5 -- PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Fold 1/5 done.
Fold 2/5 -- LIME...
    LIME: 0/36
    LIME: 30/36
Fold 2/5 -- IG...
Fold 2/5 -- PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Fold 2/5 done.
Fold 3/5 -- LIME...
    LIME: 0/35
    LIME: 30/35
Fold 3/5 -- IG...
Fold 3/5 -- PFI...
    PFI: 0/1002
    PFI: 100/1002
    PFI: 200/1002
    PFI: 300/1002
    PFI: 400/1002
    PFI: 500/1002
    PFI: 600/1002
    PFI: 700/1002
    PFI: 800/1002
    PFI: 900/1002
    PFI: 1000/1002
Fold 3/5 done.
Fold 4/5 -- LIME...
    LIME: 0/35
    LIME: 30/35
Fold 4/5 -- IG...
Fold 4/5 -- PFI...
    PFI: 0/1

In [ ]:
# --- Final summary table + save to file for download ---
comparison_results['runtimes'] = runtimes

print(json.dumps(comparison_results, indent=2, default=str))

with open('xai_comparison_results.json', 'w') as f:
    json.dump(comparison_results, f, indent=2, default=str)

print("\nSaved: xai_comparison_results.json.")

# quick readable summary table
print("\n=== SUMMARY: FAITHFULNESS (top-10 removal, higher = more faithful) ===")
for name in ['SHAP', 'LIME', 'IG', 'PFI', 'PDP']:
    print(f"  {name:6s}: {comparison_results.get(name + '_faithfulness_top10', 'N/A')}")

print("\n=== SUMMARY: GATE AGREEMENT (closer to 1 = more aligned with model's own gates) ===")
for name in ['SHAP', 'LIME', 'IG', 'PFI', 'PDP']:
    print(f"  {name:6s}: {comparison_results.get(name + '_gate_agreement', 'N/A')}")

print("\n=== SUMMARY: STABILITY (raw Jaccard / correlation-aware Jaccard, higher = more stable) ===")
for name in ['LIME', 'IG', 'PFI']:
    raw = comparison_results.get(name + '_stability_jaccard_mean', 'N/A')
    corr_aware = comparison_results.get(name + '_stability_corr_aware_mean', 'N/A')
    print(f"  {name:6s}: raw={raw}  corr_aware={corr_aware}")
print("  SHAP  : (add if you have your original per-fold SHAP saved)")

print("\n=== RUNTIMES (seconds) ===")
for name, rt in runtimes.items():
    print(f"  {name:6s}: {rt}")

{
  "SHAP_faithfulness_top5": 0.03156866999349106,
  "SHAP_faithfulness_top10": 0.03156866999349106,
  "LIME_faithfulness_top5": 0.01714037752223918,
  "LIME_faithfulness_top10": 0.046105445866782446,
  "IG_faithfulness_top5": 0.06118463875027125,
  "IG_faithfulness_top10": 0.06530700802777178,
  "PFI_faithfulness_top5": 0.03601648947711,
  "PFI_faithfulness_top10": 0.05565198524625736,
  "PDP_faithfulness_top5": 0.045454545454545525,
  "PDP_faithfulness_top10": 0.05185506617487534,
  "SHAP_gate_agreement": 0.6073138747329776,
  "SHAP_genomic_fraction": 0.9459847782770674,
  "LIME_gate_agreement": 0.5565276908862428,
  "LIME_genomic_fraction": 0.9967709621238022,
  "IG_gate_agreement": "0.5712421",
  "IG_genomic_fraction": "0.98205656",
  "PFI_gate_agreement": 0.59212910078623,
  "PFI_genomic_fraction": 0.961169552223815,
  "PDP_gate_agreement": 0.7218607544556497,
  "PDP_genomic_fraction": 0.8314378985543953,
  "model_gate_genomic_fraction": 0.553298653010045,
  "LIME_stability_jaccar

In [ ]:
# --- Per-fold SHAP (fills the missing stability comparison) ---
def compute_shap_global_per_fold(model, X_fold, nsamples=200):
    t0 = time.time()
    predict_fn = make_numpy_predict_fn(model)
    background = shap.sample(X_fold, min(30, X_fold.shape[0]))
    explainer = shap.KernelExplainer(predict_fn, background)
    shap_vals = explainer.shap_values(X_fold, nsamples=nsamples, l1_reg='num_features(50)')
    runtime = time.time() - t0
    return np.mean(np.abs(shap_vals), axis=0), runtime

per_fold_shap = []
shap_fold_runtimes = []

for i, fold_model in enumerate(fold_fusion_models):
    test_idx = fold_test_indices[i]
    sc, sg = fold_scalers[i]
    Xc_te = sc.transform(X_clinical[test_idx])
    Xg_te = sg.transform(X_genomic[test_idx])
    X_fold = np.hstack([Xc_te, Xg_te]).astype(np.float32)

    print(f"Fold {i+1}/5 -- SHAP...")
    shap_f, rt = compute_shap_global_per_fold(fold_model, X_fold)
    per_fold_shap.append(shap_f)
    shap_fold_runtimes.append(rt)
    print(f"Fold {i+1}/5 done ({rt:.1f}s).")

# --- Stability scoring, same functions as the other methods ---
mean_j, std_j = stability_score(per_fold_shap, top_k=10)
comparison_results['SHAP_stability_jaccard_mean'] = mean_j
comparison_results['SHAP_stability_jaccard_std'] = std_j

mean_jc, std_jc, n_clust = correlation_aware_stability(X_genomic, per_fold_shap, top_k=10)
comparison_results['SHAP_stability_corr_aware_mean'] = mean_jc
comparison_results['SHAP_stability_corr_aware_std'] = std_jc

comparison_results['SHAP_per_fold_runtimes'] = shap_fold_runtimes

print(f"\nSHAP stability -- raw Jaccard: {mean_j:.4f} +/- {std_j:.4f}")
print(f"SHAP stability -- corr-aware: {mean_jc:.4f} +/- {std_jc:.4f}")

# --- Re-save the complete results file ---
with open('xai_comparison_results_complete.json', 'w') as f:
    json.dump(comparison_results, f, indent=2, default=str)

print("\nSaved: xai_comparison_results_complete.json -- download and send this back.")

Fold 1/5 -- SHAP...


  0%|          | 0/36 [00:00<?, ?it/s]

Fold 1/5 done (9.1s).
Fold 2/5 -- SHAP...


  0%|          | 0/36 [00:00<?, ?it/s]

Fold 2/5 done (7.7s).
Fold 3/5 -- SHAP...


  0%|          | 0/35 [00:00<?, ?it/s]

Fold 3/5 done (9.0s).
Fold 4/5 -- SHAP...


  0%|          | 0/35 [00:00<?, ?it/s]

Fold 4/5 done (7.5s).
Fold 5/5 -- SHAP...


  0%|          | 0/35 [00:00<?, ?it/s]

Fold 5/5 done (8.3s).

SHAP stability -- raw Jaccard: 0.1178 +/- 0.0895
SHAP stability -- corr-aware: 0.1665 +/- 0.0705

Saved: xai_comparison_results_complete.json -- download and send this back.


In [ ]:
# --- IG top-20 gene ranking + beeswarm + COSMIC, for side-by-side comparison with SHAP ---

# 1. Full per-patient IG attributions (not just the mean) -- needed for the beeswarm plot
wrapped = WrappedTorchModel(model_full)
wrapped.eval()
X_t = torch.tensor(X_all.astype(np.float32), requires_grad=True)
baseline_vec = np.median(X_all, axis=0)
baseline = torch.tensor(np.tile(baseline_vec, (X_all.shape[0], 1)).astype(np.float32))
ig_full = IntegratedGradients(wrapped)
ig_attributions, _ = ig_full.attribute(X_t, baselines=baseline, n_steps=50, return_convergence_delta=True)
ig_attributions = ig_attributions.detach().numpy()

# 2. IG beeswarm (reuses shap.summary_plot -- it accepts any per-sample attribution array)
shap.summary_plot(ig_attributions, X_all, feature_names=feature_names, max_display=20, show=False)
plt.savefig('ig_beeswarm.png', dpi=500, bbox_inches='tight')
plt.close()
print("Saved: ig_beeswarm.png")

# 3. IG top-20 genes
ig_gene_scores = np.abs(ig_attributions[:, CLIN_DIM:]).mean(axis=0)
ig_top20_idx   = np.argsort(-ig_gene_scores)[:20]
ig_top20_genes = [(gene_cols[i], ig_gene_scores[i]) for i in ig_top20_idx]

print("\nTop 20 IG genes:")
for gene, val in ig_top20_genes:
    print(f"  {gene}: {val:.6f}")

# 4. COSMIC annotation for IG top-20 (reuses your existing cosmic_paad set)
print("\nTop 20 IG genes -- COSMIC PDAC driver annotation:")
print(f"{'Rank':<6}{'Gene':<15}{'Mean |IG|':<15}{'COSMIC Driver'}")
print("-" * 55)
for rank, (gene, val) in enumerate(ig_top20_genes, 1):
    is_driver = "YES" if gene in cosmic_paad else "-"
    print(f"{rank:<6}{gene:<15}{val:<15.6f}{is_driver}")

ig_overlap_cosmic = [g for g, _ in ig_top20_genes if g in cosmic_paad]
print(f"\nIG overlap with COSMIC PDAC drivers: {len(ig_overlap_cosmic)}/20 -> {ig_overlap_cosmic}")

# 5. Cross-method agreement: SHAP top-20 vs IG top-20
shap_gene_set = set(g for g, _ in top20_genes)   # from your existing SHAP cell
ig_gene_set   = set(g for g, _ in ig_top20_genes)
shared_genes  = shap_gene_set & ig_gene_set

print(f"\n=== SHAP vs IG top-20 gene overlap ===")
print(f"Shared genes ({len(shared_genes)}/20): {sorted(shared_genes)}")
print(f"SHAP-only: {sorted(shap_gene_set - ig_gene_set)}")
print(f"IG-only:   {sorted(ig_gene_set - shap_gene_set)}")

/tmp/ipykernel_11381/1168749559.py:14: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(ig_attributions, X_all, feature_names=feature_names, max_display=20, show=False)


Saved: ig_beeswarm.png

Top 20 IG genes:
  MUC2: 0.449197
  C6orf58: 0.378849
  CLCA1: 0.371074
  LY6D: 0.368243
  SPINK4: 0.307190
  KRT20: 0.257797
  GAST: 0.212547
  DLK1: 0.207972
  PGC: 0.199076
  CELP: 0.198895
  ALB: 0.164413
  KRT5: 0.154627
  CLPS: 0.135760
  CPA1: 0.132323
  REG3A: 0.131423
  CXCL9: 0.121644
  VSIG1: 0.118319
  S100A2: 0.111193
  REG3G: 0.110235
  FAM3D: 0.108964

Top 20 IG genes -- COSMIC PDAC driver annotation:
Rank  Gene           Mean |IG|      COSMIC Driver
-------------------------------------------------------
1     MUC2           0.449197       -
2     C6orf58        0.378849       -
3     CLCA1          0.371074       -
4     LY6D           0.368243       -
5     SPINK4         0.307190       -
6     KRT20          0.257797       -
7     GAST           0.212547       -
8     DLK1           0.207972       -
9     PGC            0.199076       -
10    CELP           0.198895       -
11    ALB            0.164413       -
12    KRT5           0.154627   

## Extended evaluation for the Q1 submission

**Part A (cells E1–E8): six metrics for the five XAI methods**, each answering a different "why is method A better or worse than B":

| # | Metric | Question it answers | Better |
|---|---|---|---|
| M1 | Deletion / insertion AUC (vs random features, bootstrap CI) | Does the global ranking drive the model's C-index more than random genes do? | ↑ |
| M2 | Comprehensiveness / sufficiency (per patient) | Are each patient's top features the ones that move *their* risk score? | Comp ↑, Suff ↓ |
| M3 | Model-randomisation sanity check (Adebayo 2018) | Does the explanation actually depend on the trained weights? | \|ρ\| ↓ |
| M4 | Ground-truth Precision / Recall / F1@k, AUPRC (semi-synthetic) | Does the method recover genes known to drive survival? | ↑ |
| M5 | Max-sensitivity (Yeh 2019) | Does a tiny input change flip the explanation? | ↓ |
| M6 | Complexity: normalised entropy, #features for 80% mass | How concise is the explanation? | ↓ |

Plus full-ranking stability across folds (Spearman, and the missing PDP Jaccard) and a SHAP sampling-budget sensitivity check (E8).

**Part B (cell E9): predictive metrics for the four survival models** — Harrell's C, Uno's C, time-dependent AUC at 12/24 months, integrated Brier score vs a Kaplan–Meier reference, F1 / balanced accuracy for 24-month mortality, and an out-of-fold Kaplan–Meier log-rank test.

Run after cells 0–20. Every slow step checkpoints to Drive (`extended_eval_*`), so a disconnect only costs the cell that was running. Approximate CPU time (measured end-to-end): E2 1 min, E3 0.5 min, E4 1.5 min, E5 4.5 min, E6 1.5 min, E7 seconds (about 8 min if cells 17/19 were not run this session), E8 3 min, E9 1 min — about 13 min in total.

In [ ]:
# ===== E1. Shared helpers =====
import copy
import warnings
from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from sklearn.metrics import average_precision_score
warnings.filterwarnings('ignore')

METHODS       = ['SHAP', 'LIME', 'IG', 'PFI', 'PDP']
LOCAL_METHODS = ['SHAP', 'LIME', 'IG']
E_int  = E.astype(int)
N_PAT, N_FEAT = X_all.shape
EXT_DIR = f'{CHECKPOINT_DIR}/extended_eval'
os.makedirs(EXT_DIR, exist_ok=True)
EXT = {}  # every headline number lands here and is saved to JSON in E10

# Global importances from cell 15 are not checkpointed by the original notebook -- save them now
if 'global_importances' in globals():
    save_checkpoint('extended_eval_globals', {'global_importances': global_importances,
                                              'candidates': sorted(int(c) for c in candidates),
                                              'runtimes': runtimes})
else:
    _g = load_checkpoint('extended_eval_globals')
    if _g is None:
        raise RuntimeError('global_importances not in memory and no checkpoint -- run cells 14-15 first.')
    global_importances, candidates, runtimes = _g['global_importances'], set(_g['candidates']), _g['runtimes']
cand_list = sorted(int(c) for c in candidates)


def harrell_c_many(T_, E_, R):
    """Harrell's C for every row of R (m x n). Same pairing and tie rules as
    lifelines.concordance_index(T, -risk, E) -- verified identical, including tied survival times."""
    T_ = np.asarray(T_, float); E_ = np.asarray(E_).astype(bool); R = np.atleast_2d(np.asarray(R, float))
    comp = ((T_[:, None] < T_[None, :]) | ((T_[:, None] == T_[None, :]) & ~E_[None, :])) & E_[:, None]
    n = comp.sum()
    if n == 0:
        return np.full(len(R), np.nan)
    gt = (R[:, :, None] > R[:, None, :]) & comp
    eq = (R[:, :, None] == R[:, None, :]) & comp
    return (gt.sum((1, 2)) + 0.5 * eq.sum((1, 2))) / n


def harrell_c(T_, E_, risk):
    return float(harrell_c_many(T_, E_, np.asarray(risk)[None])[0])


def shap_local(model, X_explain, X_background, nsamples=200, seed=42):
    """Per-patient KernelSHAP, same configuration as cells 11/19."""
    np.random.seed(seed)
    bg = shap.sample(X_background, min(30, len(X_background)))
    ex = shap.KernelExplainer(make_numpy_predict_fn(model), bg)
    sv = ex.shap_values(X_explain, nsamples=nsamples, l1_reg='num_features(50)', silent=True)
    return np.asarray(sv).reshape(len(X_explain), -1)


def lime_local(model, X_explain, X_train, num_samples=300, seed=42):
    """Per-patient LIME, same configuration as cell 14 (seeded here for reproducibility)."""
    ex = LimeTabularExplainer(training_data=X_train, mode='regression',
                              discretize_continuous=False, random_state=seed)
    fn = make_numpy_predict_fn(model)
    out = np.zeros(X_explain.shape, dtype=float)
    for i in range(len(X_explain)):
        e = ex.explain_instance(X_explain[i], fn, num_features=X_explain.shape[1], num_samples=num_samples)
        for j, w in e.as_map()[1]:
            out[i, j] = w
    return out


def ig_local(model, X_explain, baseline_vec=None, n_steps=50):
    """Per-patient Integrated Gradients, same configuration as cells 14/20."""
    wrapped = WrappedTorchModel(model).eval()
    if baseline_vec is None:
        baseline_vec = np.median(X_explain, axis=0)
    X_t  = torch.tensor(X_explain.astype(np.float32), requires_grad=True)
    base = torch.tensor(np.tile(baseline_vec, (len(X_explain), 1)).astype(np.float32))
    return IntegratedGradients(wrapped).attribute(X_t, baselines=base, n_steps=n_steps).detach().numpy()


def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj


def paired_tests(W, metric, higher_better):
    """W: DataFrame, rows = patients, columns = methods.
    Friedman omnibus + Holm-corrected pairwise Wilcoxon signed-rank."""
    W = W.dropna()
    fr = friedmanchisquare(*[W[c] for c in W.columns]).pvalue if W.shape[1] >= 3 else np.nan
    rows = []
    for a, b in combinations(W.columns, 2):
        d = W[a] - W[b]
        p = wilcoxon(W[a], W[b]).pvalue if np.any(d != 0) else 1.0
        rows.append({'metric': metric, 'A': a, 'B': b, 'mean_diff_A_minus_B': float(d.mean()), 'p_raw': p})
    t = pd.DataFrame(rows); t['p_holm'] = holm(t['p_raw'])
    winner = np.where((t['mean_diff_A_minus_B'] > 0) == higher_better, t['A'], t['B'])
    t['better'] = np.where(t['p_holm'] < 0.05, winner, 'n.s.')
    return fr, t


def bootstrap_pairs(boot, metric, higher_better):
    """boot: dict method -> (N_BOOT,) bootstrap replicates. Two-sided paired bootstrap p, Holm-corrected."""
    rows = []
    for a, b in combinations(boot.keys(), 2):
        d = boot[a] - boot[b]
        rows.append({'metric': metric, 'A': a, 'B': b, 'mean_diff_A_minus_B': float(d.mean()),
                     'p_raw': min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))})
    t = pd.DataFrame(rows); t['p_holm'] = holm(t['p_raw'])
    winner = np.where((t['mean_diff_A_minus_B'] > 0) == higher_better, t['A'], t['B'])
    t['better'] = np.where(t['p_holm'] < 0.05, winner, 'n.s.')
    return t


# figure style: validated colour-blind-safe categorical order, plus a distinct marker per method
METHOD_COLORS  = {'SHAP': '#2a78d6', 'LIME': '#eb6834', 'IG': '#1baf7a', 'PFI': '#eda100', 'PDP': '#e87ba4'}
METHOD_MARKERS = {'SHAP': 'o', 'LIME': 's', 'IG': '^', 'PFI': 'D', 'PDP': 'v'}

pred_full = make_numpy_predict_fn(model_full)
med_all   = np.median(X_all, axis=0).astype(np.float32)
r_orig    = pred_full(X_all)
c_orig    = harrell_c(T, E_int, r_orig)
print(f"C-index check: fast={c_orig:.6f}  lifelines={concordance_index(T, -r_orig, E):.6f}")
print("Helpers ready.")

In [ ]:
# ===== E2. M1 -- Deletion / insertion curves (global faithfulness) =====
# Extends cell 16: same model, same median replacement, but over k = 1..100 features,
# relative to deleting k RANDOM features, with patient-bootstrap 95% CIs and paired tests.
KS, N_RANDOM, N_BOOT = [1, 2, 5, 10, 20, 50, 100], 20, 1000
M_, K_ = len(METHODS), len(KS)


def _delete(idx):
    Xp = X_all.copy(); Xp[:, idx] = med_all[idx]; return pred_full(Xp)


def _insert(idx):
    Xp = np.tile(med_all, (N_PAT, 1)); Xp[:, idx] = X_all[:, idx]; return pred_full(Xp)


rs = np.random.RandomState(42)
orders  = {m: np.argsort(-global_importances[m]) for m in METHODS}
DEL     = np.stack([[_delete(orders[m][:k]) for k in KS] for m in METHODS])      # (M, K, n)
INS     = np.stack([[_insert(orders[m][:k]) for k in KS] for m in METHODS])
rnd_idx = [[rs.choice(N_FEAT, k, replace=False) for k in KS] for _ in range(N_RANDOM)]
RND_DEL = np.stack([[_delete(ix) for ix in row] for row in rnd_idx])              # (R, K, n)
RND_INS = np.stack([[_insert(ix) for ix in row] for row in rnd_idx])
STACK   = np.concatenate([DEL.reshape(M_ * K_, -1), INS.reshape(M_ * K_, -1),
                          RND_DEL.reshape(N_RANDOM * K_, -1), RND_INS.reshape(N_RANDOM * K_, -1),
                          r_orig[None]])


def m1_stats(ii):
    C  = harrell_c_many(T[ii], E_int[ii], STACK[:, ii])
    a, b, c = M_ * K_, 2 * M_ * K_, 2 * M_ * K_ + N_RANDOM * K_
    cd, ci = C[:a].reshape(M_, K_), C[a:b].reshape(M_, K_)
    rd, ri = C[b:c].reshape(N_RANDOM, K_), C[c:-1].reshape(N_RANDOM, K_)
    c0 = C[-1]
    del_drop = c0 - cd                                  # C-index drop at each k
    del_auc  = del_drop.mean(1) - (c0 - rd).mean()      # excess over random deletion
    ins_auc  = ci.mean(1) - ri.mean()                   # excess over random insertion
    return del_drop, ci, del_auc, ins_auc, (c0 - rd).mean(0), ri.mean(0)


del_drop, ins_c, del_auc, ins_auc, rnd_drop_curve, rnd_ins_curve = m1_stats(np.arange(N_PAT))
boot_del, boot_ins = np.zeros((N_BOOT, M_)), np.zeros((N_BOOT, M_))
for bi in range(N_BOOT):
    ii = rs.randint(0, N_PAT, N_PAT)
    _, _, boot_del[bi], boot_ins[bi], _, _ = m1_stats(ii)

m1 = pd.DataFrame({'method': METHODS, 'deletion_AUC_excess': del_auc,
                   'del_CI_lo': np.percentile(boot_del, 2.5, 0), 'del_CI_hi': np.percentile(boot_del, 97.5, 0),
                   'insertion_AUC_excess': ins_auc,
                   'ins_CI_lo': np.percentile(boot_ins, 2.5, 0), 'ins_CI_hi': np.percentile(boot_ins, 97.5, 0)})
for j, k in enumerate(KS):
    m1[f'drop_k{k}'] = del_drop[:, j]
m1_pairs = pd.concat([bootstrap_pairs({m: boot_del[:, i] for i, m in enumerate(METHODS)}, 'deletion_AUC', True),
                      bootstrap_pairs({m: boot_ins[:, i] for i, m in enumerate(METHODS)}, 'insertion_AUC', True)],
                     ignore_index=True)

print("Reproduction check against cell 16 (must match):")
for i, m in enumerate(METHODS):
    print(f"  {m:5s} top5 {del_drop[i, KS.index(5)]:.6f} vs {comparison_results[m + '_faithfulness_top5']:.6f} | "
          f"top10 {del_drop[i, KS.index(10)]:.6f} vs {comparison_results[m + '_faithfulness_top10']:.6f}")
print(f"\nRandom-deletion drop by k: " + ", ".join(f"k={k}:{v:.4f}" for k, v in zip(KS, rnd_drop_curve)))
print(f"PDP is scored on {len(cand_list)} candidate features; for k > {len(cand_list)} its extra features are arbitrary.")
print(m1[['method', 'deletion_AUC_excess', 'del_CI_lo', 'del_CI_hi', 'insertion_AUC_excess', 'ins_CI_lo', 'ins_CI_hi']]
      .round(4).to_string(index=False))
print("\nSignificant pairwise differences (Holm p < 0.05):")
print(m1_pairs[m1_pairs.p_holm < 0.05][['metric', 'A', 'B', 'mean_diff_A_minus_B', 'p_holm', 'better']].to_string(index=False)
      or "  none")
EXT['M1'] = m1.set_index('method').to_dict('index')
m1.to_csv(f'{EXT_DIR}/M1_deletion_insertion.csv', index=False)
m1_pairs.to_csv(f'{EXT_DIR}/M1_pairwise_bootstrap.csv', index=False)

# Figure: deletion and insertion curves
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, Y, rnd, ylab, ttl in [(axes[0], del_drop, rnd_drop_curve, 'C-index drop after deleting top-k', 'Deletion (higher = more faithful)'),
                              (axes[1], ins_c, rnd_ins_curve, 'C-index with only top-k kept', 'Insertion (higher = more faithful)')]:
    ax.plot(KS, rnd, color='#8a8a85', lw=2, ls='--', label='Random features')
    for i, m in enumerate(METHODS):
        ax.plot(KS, Y[i], color=METHOD_COLORS[m], marker=METHOD_MARKERS[m], ms=8, lw=2, label=m)
    ax.set_xscale('log'); ax.set_xticks(KS); ax.set_xticklabels(KS)
    ax.set_xlabel('k (features)'); ax.set_ylabel(ylab); ax.set_title(ttl, fontsize=10)
    ax.grid(alpha=0.25, lw=0.6); ax.spines[['top', 'right']].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=6, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout(); plt.savefig(f'{EXT_DIR}/fig_deletion_insertion.png', dpi=500, bbox_inches='tight'); plt.close()
print(f"\nSaved: {EXT_DIR}/fig_deletion_insertion.png")

In [ ]:
# ===== E3. M2 -- Comprehensiveness & sufficiency (per-patient local faithfulness; DeYoung et al., 2020) =====
# Comprehensiveness: |risk change| when the patient's own top-k features are set to the cohort median  (higher = better)
# Sufficiency:       |risk change| when ONLY the top-k features keep their values                     (lower = better)
# Averaged over k = 5, 10, 20, in units of the cohort risk SD. Global methods use one ranking for every patient.
KS_LOCAL = [5, 10, 20]
ck = load_checkpoint('extended_eval_local_full')
if ck is None:
    t0 = time.time()
    ck = {'LIME': lime_local(model_full, X_all, X_all, num_samples=300),
          'IG':   ig_local(model_full, X_all)}
    save_checkpoint('extended_eval_local_full', ck)
    print(f"Local LIME/IG recomputed in {time.time() - t0:.0f}s")
LOCAL_FULL = {'SHAP': np.asarray(shap_values).reshape(N_PAT, -1), 'LIME': ck['LIME'], 'IG': ck['IG']}

sd_r = r_orig.std()
comp, suff = {}, {}
for m in METHODS:
    imp = np.abs(LOCAL_FULL[m]) if m in LOCAL_METHODS else np.tile(global_importances[m], (N_PAT, 1))
    c_acc, s_acc = np.zeros(N_PAT), np.zeros(N_PAT)
    for k in KS_LOCAL:
        top = np.argsort(-imp, axis=1)[:, :k]
        Xd = X_all.copy();               np.put_along_axis(Xd, top, med_all[top], axis=1)
        Xk = np.tile(med_all, (N_PAT, 1)); np.put_along_axis(Xk, top, np.take_along_axis(X_all, top, 1), axis=1)
        c_acc += np.abs(r_orig - pred_full(Xd)) / sd_r
        s_acc += np.abs(r_orig - pred_full(Xk)) / sd_r
    comp[m], suff[m] = c_acc / len(KS_LOCAL), s_acc / len(KS_LOCAL)

comp_df, suff_df = pd.DataFrame(comp), pd.DataFrame(suff)
fr_c, pw_c = paired_tests(comp_df, 'comprehensiveness', True)
fr_s, pw_s = paired_tests(suff_df, 'sufficiency', False)
m2 = pd.DataFrame({'method': METHODS,
                   'comprehensiveness': comp_df.mean().values, 'comp_sd': comp_df.std().values,
                   'sufficiency': suff_df.mean().values, 'suff_sd': suff_df.std().values})
print(m2.round(4).to_string(index=False))
print(f"\nFriedman p: comprehensiveness {fr_c:.2e}, sufficiency {fr_s:.2e}  (n = {N_PAT} patients)")
m2_pairs = pd.concat([pw_c, pw_s], ignore_index=True)
print(m2_pairs[m2_pairs.p_holm < 0.05][['metric', 'A', 'B', 'mean_diff_A_minus_B', 'p_holm', 'better']].to_string(index=False)
      or "  none significant")
EXT['M2'] = {**m2.set_index('method').to_dict('index'), 'friedman_p_comp': fr_c, 'friedman_p_suff': fr_s}
m2.to_csv(f'{EXT_DIR}/M2_comp_suff.csv', index=False); m2_pairs.to_csv(f'{EXT_DIR}/M2_pairwise_wilcoxon.csv', index=False)

In [ ]:
# ===== E4. M3 -- Model-parameter randomisation sanity check (Adebayo et al., 2018) =====
# Re-initialise every Linear layer, re-run each method with identical settings, and correlate the
# trained-model ranking with the random-model ranking. A method that passes gives |rho| near 0.


def randomise_model(model, seed=0):
    torch.manual_seed(seed)
    rm = copy.deepcopy(model)
    for mod in rm.modules():
        if isinstance(mod, nn.Linear):
            mod.reset_parameters()
    return rm.eval()


ck = load_checkpoint('extended_eval_sanity')
if ck is None:
    t0 = time.time()
    model_rand = randomise_model(model_full)
    rand_glob = {'SHAP': np.abs(shap_local(model_rand, X_all, X_all)).mean(0),
                 'LIME': np.abs(lime_local(model_rand, X_all, X_all, num_samples=300)).mean(0),
                 'IG':   np.abs(ig_local(model_rand, X_all)).mean(0)}
    rand_glob['PFI'], _ = compute_pfi_global(model_rand, X_all, T, E)
    rand_glob['PDP'], _ = compute_pdp_global(model_rand, X_all, cand_list)
    ck = {'rand_glob': rand_glob, 'c_rand': harrell_c(T, E_int, make_numpy_predict_fn(model_rand)(X_all))}
    save_checkpoint('extended_eval_sanity', ck)
    print(f"Randomised-model attributions computed in {time.time() - t0:.0f}s")

rows = []
for m in METHODS:
    g, gr = global_importances[m], ck['rand_glob'][m]
    mask = np.isin(np.arange(N_FEAT), cand_list) if m == 'PDP' else np.ones(N_FEAT, bool)
    rho = spearmanr(g[mask], gr[mask]).correlation
    top10 = len(set(np.argsort(-g)[:10]) & set(np.argsort(-gr)[:10])) / 10
    rows.append({'method': m, 'spearman_rho': rho, 'abs_rho': abs(rho), 'top10_overlap': top10})
m3 = pd.DataFrame(rows)
print(f"Randomised model C-index = {ck['c_rand']:.3f} (trained: {c_orig:.3f})")
print(m3.round(4).to_string(index=False))
m3['top10_overlap_chance'] = [10 / len(cand_list) if m == 'PDP' else 10 / N_FEAT for m in METHODS]
print(m3[['method', 'top10_overlap_chance']].round(3).to_string(index=False))
print("Pass = |rho| near 0 and top-10 overlap near chance. A high |rho| means the ranking reflects the input data,\n"
      "not what the model learned (Adebayo et al. found gradient x input-type methods can fail this).")
EXT['M3'] = {**m3.set_index('method').to_dict('index'), 'random_model_cindex': ck['c_rand']}
m3.to_csv(f'{EXT_DIR}/M3_sanity.csv', index=False)

In [ ]:
# ===== E5. M4 -- Ground-truth recovery on semi-synthetic outcomes (Precision / Recall / F1@k, AUPRC) =====
# The real TCGA-PAAD expression matrix is kept (so the true co-expression structure is kept); only the survival
# times are replaced by Weibull times driven by 10 known genes (8 linear + 1 interacting pair), with the
# cohort's median survival and censoring rate. Same model class, training and explainer settings as the paper.
N_CAUSAL, SYN_REPS, SYN_EFFECT = 10, 3, 1.5
G_corr = np.nan_to_num(np.corrcoef(X_all[:, CLIN_DIM:].T))


def simulate_outcome(seed):
    rsi = np.random.RandomState(seed)
    G = X_all[:, CLIN_DIM:]
    sd = G.std(0)
    eligible = np.where((sd > 0.5) & (sd < 2.0))[0]
    if len(eligible) < 50:
        eligible = np.arange(G.shape[1])
    S = np.sort(rsi.choice(eligible, N_CAUSAL, replace=False))
    lin, (a, b) = S[:-2], S[-2:]
    beta = rsi.uniform(0.5, 1.0, len(lin)) * rsi.choice([-1, 1], len(lin))
    eta = G[:, lin] @ beta + 1.5 * np.tanh(G[:, a] * G[:, b])
    eta = SYN_EFFECT * (eta - eta.mean()) / eta.std()
    nu = 1.5; lam = np.log(2) / np.median(T) ** nu
    Tt = (-np.log(rsi.uniform(size=N_PAT)) / (lam * np.exp(eta))) ** (1 / nu)
    target, u = 1 - E_int.mean(), rsi.uniform(size=N_PAT)
    lo, hi = 1e-3, 50 * Tt.max()
    for _ in range(60):  # bisection: censoring upper bound that reproduces the real censoring rate
        cmax = (lo + hi) / 2
        lo, hi = (cmax, hi) if np.mean(u * cmax < Tt) > target else (lo, cmax)
    Cn = u * cmax
    return np.minimum(Tt, Cn).astype(np.float32), (Tt <= Cn).astype(np.float32), S + CLIN_DIM, eta


def fit_gated(Ts, Es, idx):
    torch.manual_seed(42)
    m = train_model(GatedFusionModel(), X_all[idx, :CLIN_DIM], X_all[idx, CLIN_DIM:], Ts[idx], Es[idx])
    return m.eval()


def recovery_scores(g, S):
    y = np.zeros(N_FEAT, int); y[S] = 1
    out = {'AUPRC': average_precision_score(y, g)}
    for k in (10, 20):
        sel = np.argsort(-g)[:k]; tp = len(set(sel) & set(S))
        p_, r_ = tp / k, tp / len(S)
        out[f'precision@{k}'], out[f'recall@{k}'] = p_, r_
        out[f'F1@{k}'] = 2 * p_ * r_ / (p_ + r_) if tp else 0.0
    # correlation-tolerant F1@10: a selected gene with |r| > 0.8 to a causal gene counts as a hit
    sel_g, Sg = np.argsort(-g)[:10], S - CLIN_DIM
    sel_g = sel_g[sel_g >= CLIN_DIM] - CLIN_DIM
    if len(sel_g):
        p_c = np.mean([np.abs(G_corr[j, Sg]).max() > 0.8 for j in sel_g]) * len(sel_g) / 10
        r_c = np.mean([np.abs(G_corr[s, sel_g]).max() > 0.8 for s in Sg])
        out['F1@10_corr_tolerant'] = 2 * p_c * r_c / (p_c + r_c) if p_c + r_c else 0.0
    else:
        out['F1@10_corr_tolerant'] = 0.0
    return out


syn_rows = load_checkpoint('extended_eval_synthetic') or []
for rep in range(len(syn_rows) // len(METHODS), SYN_REPS):
    t0, seed = time.time(), 1000 + rep
    Ts, Es, S, eta = simulate_outcome(seed)
    # learnability check: does the same architecture learn the planted signal out-of-fold?
    cv_c, or_c = [], []
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=42).split(X_all, Es):
        mm = fit_gated(Ts, Es, tr)
        with torch.no_grad():
            rr = mm(torch.tensor(X_all[te, :CLIN_DIM]), torch.tensor(X_all[te, CLIN_DIM:])).numpy()
        cv_c.append(harrell_c(Ts[te], Es[te].astype(int), rr)); or_c.append(harrell_c(Ts[te], Es[te].astype(int), eta[te]))
    msyn = fit_gated(Ts, Es, np.arange(N_PAT))
    g = {'SHAP': np.abs(shap_local(msyn, X_all, X_all)).mean(0),
         'LIME': np.abs(lime_local(msyn, X_all, X_all, num_samples=300)).mean(0),
         'IG':   np.abs(ig_local(msyn, X_all)).mean(0)}
    g['PFI'], _ = compute_pfi_global(msyn, X_all, Ts, Es)
    cand_syn = sorted(set().union(*[set(np.argsort(-g[mm_])[:25]) for mm_ in ['SHAP', 'LIME', 'IG', 'PFI']]))
    g['PDP'], _ = compute_pdp_global(msyn, X_all, cand_syn)
    for m in METHODS:
        syn_rows.append({'rep': rep + 1, 'method': m, **recovery_scores(g[m], S),
                         'model_cv_C': np.mean(cv_c), 'oracle_cv_C': np.mean(or_c)})
    save_checkpoint('extended_eval_synthetic', syn_rows)
    print(f"Rep {rep + 1}/{SYN_REPS} ({time.time() - t0:.0f}s): model CV C = {np.mean(cv_c):.3f}, "
          f"oracle C = {np.mean(or_c):.3f}; F1@10 " +
          ", ".join(f"{r['method']}={r['F1@10']:.2f}" for r in syn_rows[-len(METHODS):]))

syn_df = pd.DataFrame(syn_rows)
m4 = syn_df.groupby('method', sort=False)[['F1@10', 'F1@20', 'precision@20', 'recall@20', 'F1@10_corr_tolerant', 'AUPRC']] \
           .agg(['mean', 'std']).reindex(METHODS)
print(f"\nPrevalence baseline: AUPRC of a random ranking = {N_CAUSAL / N_FEAT:.4f}; "
      f"F1@10 of a random top-10 ~ {N_CAUSAL / N_FEAT:.3f}")
print(f"Learnability: model CV C = {syn_df.model_cv_C.mean():.3f} vs oracle {syn_df.oracle_cv_C.mean():.3f}")
print("Note: at k = 10 = number of causal genes, precision@10 = recall@10 = F1@10.")
print(m4.round(3).to_string())
EXT['M4'] = {m: {f'{a}_{b}': float(m4.loc[m, (a, b)]) for a, b in m4.columns} for m in METHODS}
EXT['M4']['learnability'] = {'model_cv_C': float(syn_df.model_cv_C.mean()), 'oracle_cv_C': float(syn_df.oracle_cv_C.mean())}
syn_df.to_csv(f'{EXT_DIR}/M4_ground_truth_reps.csv', index=False)

In [ ]:
# ===== E6. M5 -- Local robustness: max-sensitivity (Yeh et al., 2019) =====
# For 30 patients, add Gaussian noise of 5% of each feature's IQR (5 draws) and measure
#   max_j ||a(x + noise_j) - a(x)|| / ||a(x)||        (lower = more robust).
# The "re-run floor" repeats the same computation with NO noise: for stochastic methods (SHAP, LIME)
# it shows how much of the sensitivity is pure sampling noise. IG is deterministic, so its floor is 0.
N_ROB, N_PERT, SIGMA = 30, 5, 0.05
ck = load_checkpoint('extended_eval_robustness')
if ck is None:
    t0 = time.time()
    rsr = np.random.RandomState(42)
    sub = rsr.choice(N_PAT, N_ROB, replace=False)
    iqr = np.subtract(*np.percentile(X_all, [75, 25], axis=0))
    X0  = np.repeat(X_all[sub], N_PERT, axis=0)
    Xp  = (X0 + rsr.normal(0, 1, X0.shape) * SIGMA * iqr).astype(np.float32)
    ck = {'sub': sub}
    for tag, XX in [('pert', Xp), ('rerun', X0)]:
        ck[tag] = {'SHAP': shap_local(model_full, XX, X_all, seed=7),
                   'LIME': lime_local(model_full, XX, X_all, num_samples=300, seed=7),
                   'IG':   ig_local(model_full, XX, baseline_vec=med_all)}
    save_checkpoint('extended_eval_robustness', ck)
    print(f"Robustness attributions computed in {time.time() - t0:.0f}s")

sub = ck['sub']
sens, floor = {}, {}
for m in LOCAL_METHODS:
    a0 = LOCAL_FULL[m][sub]
    for tag, store in [('pert', sens), ('rerun', floor)]:
        ap = ck[tag][m].reshape(N_ROB, N_PERT, -1)
        store[m] = np.linalg.norm(ap - a0[:, None], axis=2).max(1) / (np.linalg.norm(a0, axis=1) + 1e-12)
sens_df, floor_df = pd.DataFrame(sens), pd.DataFrame(floor)
fr_r, pw_r = paired_tests(sens_df, 'max_sensitivity', False)
m5 = pd.DataFrame({'method': LOCAL_METHODS, 'max_sensitivity_mean': sens_df.mean().values,
                   'max_sensitivity_median': sens_df.median().values, 'rerun_floor_mean': floor_df.mean().values})
print(m5.round(4).to_string(index=False))
print(f"Friedman p = {fr_r:.2e} (n = {N_ROB} patients)")
print(pw_r[['A', 'B', 'mean_diff_A_minus_B', 'p_holm', 'better']].to_string(index=False))
print("PFI and PDP are global methods (one ranking for the cohort), so per-patient robustness does not apply.")
EXT['M5'] = {**m5.set_index('method').to_dict('index'), 'friedman_p': fr_r}
m5.to_csv(f'{EXT_DIR}/M5_robustness.csv', index=False); pw_r.to_csv(f'{EXT_DIR}/M5_pairwise_wilcoxon.csv', index=False)

In [ ]:
# ===== E7. M6 -- Complexity, plus full-ranking stability across the 5 CV folds =====


def complexity(g, support=None):
    a = np.abs(g if support is None else g[support]); q = a / a.sum(); qn = q[q > 0]
    H = -(qn * np.log(qn)).sum()
    n80 = int(np.searchsorted(np.cumsum(np.sort(q)[::-1]), 0.8) + 1)
    return H / np.log(len(a)), float(np.exp(H)), n80


m6 = []
for m in METHODS:
    h, eff, n80 = complexity(global_importances[m], cand_list if m == 'PDP' else None)
    m6.append({'method': m, 'norm_entropy': h, 'effective_features': eff, 'features_for_80pct': n80})
m6 = pd.DataFrame(m6)
print("M6 complexity (lower = more concise). PDP is normalised over its candidate set only.")
print("Caution: SHAP uses l1_reg='num_features(50)', which forces <= 50 non-zero features per patient.")
print(m6.round(3).to_string(index=False))
EXT['M6'] = m6.set_index('method').to_dict('index')
m6.to_csv(f'{EXT_DIR}/M6_complexity.csv', index=False)

# --- Per-fold global importances: reuse cells 17/19 if still in memory, otherwise recompute (same settings)
ck = load_checkpoint('extended_eval_fold_globals')
if ck is None:
    have = all(v in globals() for v in ['per_fold_shap', 'per_fold_lime', 'per_fold_ig', 'per_fold_pfi'])
    PF = {m: [] for m in METHODS}
    for i, fold_model in enumerate(fold_fusion_models):
        te = fold_test_indices[i]; sc, sg = fold_scalers[i]
        X_fold = np.hstack([sc.transform(X_clinical[te]), sg.transform(X_genomic[te])]).astype(np.float32)
        if have:
            PF['SHAP'].append(per_fold_shap[i]); PF['LIME'].append(per_fold_lime[i])
            PF['IG'].append(per_fold_ig[i]);     PF['PFI'].append(per_fold_pfi[i])
        else:
            print(f"Fold {i + 1}/5: recomputing SHAP/LIME/IG/PFI ...")
            PF['SHAP'].append(compute_shap_global_per_fold(fold_model, X_fold)[0])
            PF['LIME'].append(compute_lime_global(fold_model, X_fold, feature_names, num_samples=150)[0])
            PF['IG'].append(compute_ig_global(fold_model, X_fold)[0])
            PF['PFI'].append(compute_pfi_global(fold_model, X_fold, T[te], E[te], n_repeats=5)[0])
        cand_f = sorted(set().union(*[set(np.argsort(-PF[mm_][-1])[:25]) for mm_ in ['SHAP', 'LIME', 'IG', 'PFI']]))
        PF['PDP'].append(compute_pdp_global(fold_model, X_fold, cand_f)[0])
    ck = PF
    save_checkpoint('extended_eval_fold_globals', ck)
PF = ck

stab = []
for m in METHODS:
    rhos = [spearmanr(PF[m][a], PF[m][b]).correlation for a, b in combinations(range(5), 2)] if m != 'PDP' else [np.nan]
    j_raw, _ = stability_score(PF[m], top_k=10)
    j_cor, _, _ = correlation_aware_stability(X_genomic, PF[m], top_k=10)
    stab.append({'method': m, 'rank_spearman_mean': np.nanmean(rhos), 'rank_spearman_sd': np.nanstd(rhos),
                 'jaccard10_raw': j_raw, 'jaccard10_corr_aware': j_cor})
stab = pd.DataFrame(stab)
print("\nFull-ranking stability across folds (Spearman over all 1,002 features; PDP has no full ranking).")
print("Jaccard values for SHAP/LIME/IG/PFI should match the locked cell 17/19 results; PDP is new.")
print(stab.round(4).to_string(index=False))
EXT['stability'] = stab.set_index('method').to_dict('index')
stab.to_csv(f'{EXT_DIR}/S_rank_stability.csv', index=False)

In [ ]:
# ===== E8. Sensitivity check: is SHAP's lower faithfulness an artefact of its sampling budget? =====
# Cells 11/19 give KernelSHAP 200 coalitions for 1,002 features with l1_reg='num_features(50)'.
# KernelExplainer's own default for 1,002 features is 2*M + 2048 = 4,052. A reviewer will ask whether
# "IG beats SHAP" survives a larger budget, so we re-run SHAP with 2,048 coalitions and no L1 truncation.
SHAP_HI_NSAMPLES = 2048
ck = load_checkpoint('extended_eval_shap_hi')
if ck is None:
    t0 = time.time()
    np.random.seed(42)
    ex_hi = shap.KernelExplainer(pred_full, shap.sample(X_all, 30))
    sv_hi = np.asarray(ex_hi.shap_values(X_all, nsamples=SHAP_HI_NSAMPLES, l1_reg=False, silent=True)).reshape(N_PAT, -1)
    ck = {'shap_hi': sv_hi, 'runtime': time.time() - t0}
    save_checkpoint('extended_eval_shap_hi', ck)
g_hi = np.abs(ck['shap_hi']).mean(0)
o_hi = np.argsort(-g_hi)
o_ig = np.argsort(-global_importances['IG'])
del_hi = np.array([c_orig - harrell_c(T, E_int, _delete(o_hi[:k])) for k in KS])
rnd_mean = rnd_drop_curve.mean()
shap_hi_res = {
    'nsamples': SHAP_HI_NSAMPLES, 'runtime_s': ck['runtime'],
    'faithfulness_top5':  faithfulness_score(model_full, X_all, g_hi, T, E, top_k=5),
    'faithfulness_top10': faithfulness_score(model_full, X_all, g_hi, T, E, top_k=10),
    'deletion_AUC_excess': del_hi.mean() - rnd_mean,
    'spearman_vs_SHAP200': spearmanr(g_hi, global_importances['SHAP']).correlation,
    'spearman_vs_IG': spearmanr(g_hi, global_importances['IG']).correlation,
    'top20_genes_overlap_with_IG': len(set(o_hi[o_hi >= CLIN_DIM][:20]) & set(o_ig[o_ig >= CLIN_DIM][:20])),
    'top_gene': feature_names[o_hi[0]],
    'top5_genes': [feature_names[i] for i in o_hi[o_hi >= CLIN_DIM][:5]],
}
print(f"SHAP @ {SHAP_HI_NSAMPLES} coalitions ({ck['runtime']:.0f}s):")
for k_, v_ in shap_hi_res.items():
    print(f"  {k_}: {v_}")
print(f"Compare: SHAP@200 faithfulness top10 = {comparison_results['SHAP_faithfulness_top10']:.4f}, "
      f"IG = {comparison_results['IG_faithfulness_top10']:.4f}; "
      f"deletion AUC excess SHAP@200 = {del_auc[0]:.4f}, IG = {del_auc[2]:.4f}")
gap_200 = comparison_results['IG_faithfulness_top10'] - comparison_results['SHAP_faithfulness_top10']
gap_hi  = comparison_results['IG_faithfulness_top10'] - shap_hi_res['faithfulness_top10']
print(f"IG-minus-SHAP faithfulness gap: {gap_200:.4f} at 200 coalitions -> {gap_hi:.4f} at {SHAP_HI_NSAMPLES}.")
print("If the gap shrinks a lot, the paper must say SHAP's deficit is partly budget-dependent.")
EXT['SHAP_budget_sensitivity'] = shap_hi_res

In [ ]:
# ===== E9. Part B -- Predictive metrics for the four survival models (out-of-fold, same 5 folds) =====
# Harrell's C, Uno's C (IPCW, tau = 36 mo), time-dependent AUC at 12 / 24 mo, integrated Brier score
# (Breslow survival curves) vs a Kaplan-Meier "no-model" reference, F1 / balanced accuracy / sensitivity /
# specificity for 24-month mortality (threshold = training-fold median risk), and an OUT-OF-FOLD log-rank test.
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc, integrated_brier_score
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score

HORIZONS, TAU, CLS_H, N_BOOT_P = [12, 24], 36.0, 24, 2000
MODEL_SPECS = [('Clinical MLP', ClinicalMLP), ('Genomic MLP', GenomicMLP),
               ('Gated fusion', GatedFusionModel), ('Concat fusion', ConcatFusionModel)]
folds = list(skf.split(X_clinical, E))


def risk_of(model, Xc, Xg):
    with torch.no_grad():
        if isinstance(model, ClinicalMLP):
            return model(torch.tensor(Xc)).numpy()
        if isinstance(model, GenomicMLP):
            return model(torch.tensor(Xg)).numpy()
        return model(torch.tensor(Xc), torch.tensor(Xg)).numpy()


ck = load_checkpoint('extended_eval_oof_preds')
if ck is None:
    t0 = time.time()
    ck = {name: {'tr': [], 'te': []} for name, _ in MODEL_SPECS}
    for f, (tr, te) in enumerate(folds):
        sc = RobustScaler(); sg = RobustScaler()
        Xc_tr = sc.fit_transform(X_clinical[tr]).astype(np.float32); Xc_te = sc.transform(X_clinical[te]).astype(np.float32)
        Xg_tr = sg.fit_transform(X_genomic[tr]).astype(np.float32);  Xg_te = sg.transform(X_genomic[te]).astype(np.float32)
        for name, cls in MODEL_SPECS:
            torch.manual_seed(42)  # identical seeding to cells 8-9 -> identical models
            mdl = train_model(cls(), Xc_tr, Xg_tr, T[tr], E[tr]).eval()
            ck[name]['tr'].append(risk_of(mdl, Xc_tr, Xg_tr)); ck[name]['te'].append(risk_of(mdl, Xc_te, Xg_te))
    save_checkpoint('extended_eval_oof_preds', ck)
    print(f"Retrained 4 models x 5 folds in {time.time() - t0:.0f}s")
OOF = ck

locked = {'Clinical MLP': results_cv['clinical'], 'Genomic MLP': results_cv['genomic'],
          'Gated fusion': results_cv['fusion'], 'Concat fusion': globals().get('concat_scores')}
print("Reproduction check vs cells 8-9 (per-fold C-index, should match to ~1e-6):")
for name, _ in MODEL_SPECS:
    if locked[name] is None:
        print(f"  {name:13s} (cell 9 not run this session -- skipped)"); continue
    mine = [harrell_c(T[te], E_int[te], OOF[name]['te'][f]) for f, (tr, te) in enumerate(folds)]
    print(f"  {name:13s} max |diff| = {np.max(np.abs(np.array(mine) - np.array(locked[name]))):.2e}")


def surv_eval(T_te, E_te, T_tr):
    """sksurv's IPCW censoring curve is fit on the training fold and undefined past its last time point.
    Test patients followed longer than that are alive at every evaluation time used here (<= 36 months),
    so capping their time just below the training maximum leaves every metric unchanged."""
    return Surv.from_arrays(E_te.astype(bool), np.minimum(T_te, T_tr.max() - 1e-3))


def breslow_survival(r_tr, T_tr, E_tr, r_te, times):
    c = r_tr.max(); er = np.exp(r_tr - c)
    ev = np.unique(T_tr[E_tr == 1])
    H = np.cumsum([((T_tr == t) & (E_tr == 1)).sum() / er[T_tr >= t].sum() for t in ev])
    H_at = np.array([H[ev <= t][-1] if np.any(ev <= t) else 0.0 for t in times])
    return np.exp(-np.outer(np.exp(r_te - c), H_at))


pred_rows, oof_cls = [], {}
for name, _ in MODEL_SPECS:
    per = {'harrell': [], 'uno': [], 'auc12': [], 'auc24': [], 'ibs': [], 'ibs_km': []}
    y_true, y_pred, grp_T, grp_E, grp_high = [], [], [], [], []
    for f, (tr, te) in enumerate(folds):
        r_tr, r_te = OOF[name]['tr'][f], OOF[name]['te'][f]
        y_tr, y_te = Surv.from_arrays(E[tr].astype(bool), T[tr]), surv_eval(T[te], E[te], T[tr])
        per['harrell'].append(harrell_c(T[te], E_int[te], r_te))
        per['uno'].append(concordance_index_ipcw(y_tr, y_te, r_te, tau=TAU)[0])
        aucs, _ = cumulative_dynamic_auc(y_tr, y_te, r_te, times=HORIZONS)
        per['auc12'].append(aucs[0]); per['auc24'].append(aucs[1])
        times = np.linspace(np.percentile(T[te], 10), min(np.percentile(T[te], 80), T[tr].max() - 0.01), 15)
        per['ibs'].append(integrated_brier_score(y_tr, y_te, breslow_survival(r_tr, T[tr], E_int[tr], r_te, times), times))
        km = KaplanMeierFitter().fit(T[tr], E[tr])
        per['ibs_km'].append(integrated_brier_score(y_tr, y_te, np.tile(km.survival_function_at_times(times).values, (len(te), 1)), times))
        thr = np.median(r_tr); high = r_te >= thr
        grp_T.append(T[te]); grp_E.append(E[te]); grp_high.append(high)
        known = ((E_int[te] == 1) & (T[te] <= CLS_H)) | (T[te] > CLS_H)      # excludes censored before 24 mo
        y_true.append(((E_int[te] == 1) & (T[te] <= CLS_H))[known].astype(int)); y_pred.append(high[known].astype(int))
    yt, yp = np.concatenate(y_true), np.concatenate(y_pred)
    gT, gE, gH = np.concatenate(grp_T), np.concatenate(grp_E), np.concatenate(grp_high)
    lr = logrank_test(gT[gH], gT[~gH], gE[gH], gE[~gH])
    oof_cls[name] = (yt, yp)
    row = {'model': name}
    for k_, v_ in per.items():
        row[f'{k_}_mean'], row[f'{k_}_sd'] = np.mean(v_), np.std(v_)
    row.update({'F1@24': f1_score(yt, yp), 'bal_acc@24': balanced_accuracy_score(yt, yp),
                'sensitivity@24': recall_score(yt, yp, pos_label=1), 'specificity@24': recall_score(yt, yp, pos_label=0),
                'n_classified@24': len(yt), 'logrank_p_oof': lr.p_value})
    pred_rows.append(row)
pred = pd.DataFrame(pred_rows)

# Paired, fold-stratified bootstrap of mean fold Harrell's C: each model vs gated fusion
rsb = np.random.RandomState(42)
names = [n for n, _ in MODEL_SPECS]
boot = np.zeros((N_BOOT_P, len(names)))
for bi in range(N_BOOT_P):
    vals = []
    for f, (tr, te) in enumerate(folds):
        ii = rsb.randint(0, len(te), len(te))
        vals.append(harrell_c_many(T[te][ii], E_int[te][ii], np.stack([OOF[n]['te'][f][ii] for n in names])))
    boot[bi] = np.nanmean(vals, axis=0)
gi = names.index('Gated fusion')
pred['dC_vs_gated'] = pred['harrell_mean'] - pred.loc[gi, 'harrell_mean']
pred['dC_CI_lo'] = [np.percentile(boot[:, j] - boot[:, gi], 2.5) for j in range(len(names))]
pred['dC_CI_hi'] = [np.percentile(boot[:, j] - boot[:, gi], 97.5) for j in range(len(names))]
pred['dC_p'] = [np.nan if j == gi else min(1, 2 * min(((boot[:, j] - boot[:, gi]) <= 0).mean(),
                                                      ((boot[:, j] - boot[:, gi]) >= 0).mean())) for j in range(len(names))]

pd.set_option('display.width', 250)
print("\n=== Predictive performance (5-fold out-of-fold; mean +/- SD across folds) ===")
show = ['model', 'harrell_mean', 'harrell_sd', 'uno_mean', 'auc12_mean', 'auc24_mean', 'ibs_mean', 'ibs_km_mean',
        'F1@24', 'bal_acc@24', 'sensitivity@24', 'specificity@24', 'logrank_p_oof', 'dC_vs_gated', 'dC_CI_lo', 'dC_CI_hi', 'dC_p']
print(pred[show].round(4).to_string(index=False))
print(f"\nIBS: lower is better; a model is only useful if its IBS is below the Kaplan-Meier reference "
      f"(ibs_km_mean = {pred.ibs_km_mean.iloc[0]:.4f}).")
print(f"td-AUC and F1 at 0.5 / balanced accuracy at 0.5 = chance.")
print(f"\n!! Cell 13 log-rank p was computed IN-SAMPLE (model_full trained on all 177 patients, then used to split the "
      f"same patients) and is optimistic.\n   Out-of-fold log-rank p for the gated fusion model = {pred.loc[gi, 'logrank_p_oof']:.4g}. "
      f"Report this one in the paper.")
EXT['predictive'] = pred.set_index('model').to_dict('index')
pred.to_csv(f'{EXT_DIR}/B_predictive_metrics.csv', index=False)

# OOF Kaplan-Meier figure for the gated fusion model
gT, gE, gH = [], [], []
for f, (tr, te) in enumerate(folds):
    thr = np.median(OOF['Gated fusion']['tr'][f])
    gT.append(T[te]); gE.append(E[te]); gH.append(OOF['Gated fusion']['te'][f] >= thr)
gT, gE, gH = map(np.concatenate, (gT, gE, gH))
plt.figure(figsize=(7, 5)); ax = plt.gca()
KaplanMeierFitter().fit(gT[gH], gE[gH], label=f'High risk (n={gH.sum()})').plot_survival_function(ax=ax, color='#eb6834')
KaplanMeierFitter().fit(gT[~gH], gE[~gH], label=f'Low risk (n={(~gH).sum()})').plot_survival_function(ax=ax, color='#2a78d6')
ax.set_title(f'Out-of-fold Kaplan-Meier, gated fusion (log-rank p = {pred.loc[gi, "logrank_p_oof"]:.3g})', fontsize=10)
ax.set_xlabel('Overall survival (months)'); ax.set_ylabel('Survival probability'); ax.spines[['top', 'right']].set_visible(False)
plt.savefig(f'{EXT_DIR}/fig_km_oof.png', dpi=300, bbox_inches='tight'); plt.close()
print(f"Saved: {EXT_DIR}/fig_km_oof.png")

In [ ]:
# ===== E10. Summary: headline table, per-metric ranks, LaTeX, JSON =====
H = pd.DataFrame(index=METHODS)
H['Faith@10']   = [comparison_results[f'{m}_faithfulness_top10'] for m in METHODS]
H['Del. AUC']   = m1.set_index('method')['deletion_AUC_excess']
H['Ins. AUC']   = m1.set_index('method')['insertion_AUC_excess']
H['Comp.']      = m2.set_index('method')['comprehensiveness']
H['Suff.']      = m2.set_index('method')['sufficiency']
H['Sanity |rho|'] = m3.set_index('method')['abs_rho']
H['F1@10']      = m4[('F1@10', 'mean')]
H['AUPRC']      = m4[('AUPRC', 'mean')]
H['Max-sens.']  = m5.set_index('method')['max_sensitivity_mean']
H['Entropy']    = m6.set_index('method')['norm_entropy']
H['Rank rho']   = stab.set_index('method')['rank_spearman_mean']
H['Jaccard@10'] = stab.set_index('method')['jaccard10_raw']
H['Gate agr.']  = [float(comparison_results[f'{m}_gate_agreement']) for m in METHODS]
H['Runtime (s)'] = [runtimes[m] for m in METHODS]
HIGHER = {'Faith@10': True, 'Del. AUC': True, 'Ins. AUC': True, 'Comp.': True, 'Suff.': False, 'Sanity |rho|': False,
          'F1@10': True, 'AUPRC': True, 'Max-sens.': False, 'Entropy': False, 'Rank rho': True, 'Jaccard@10': True,
          'Gate agr.': True, 'Runtime (s)': False}

R = pd.DataFrame(index=METHODS)
for c in H.columns:
    v = H[c].astype(float); ok = v.notna()
    R.loc[ok, c] = rankdata(-v[ok] if HIGHER[c] else v[ok], method='min')
R['mean rank'] = R.mean(1); R['#wins'] = (R.drop(columns='mean rank') == 1).sum(1)
print("=== XAI headline table ===");     print(H.round(4).to_string())
print("\n=== Ranks (1 = best on that metric) ==="); print(R.sort_values('mean rank').to_string())

TEX_NAME = {'Sanity |rho|': 'Sanity $|\\rho|$', 'Rank rho': 'Rank $\\rho$', 'Runtime (s)': 'Time (s)'}


def latex_table(df, higher, caption, label, fmt='{:.3f}', first_col='Method', foot=None):
    best = {c: (df[c].astype(float).idxmax() if higher[c] else df[c].astype(float).idxmin())
            for c in df.columns if df[c].notna().any()}
    UP, DOWN = '$\\uparrow$', '$\\downarrow$'
    hdr = ' & '.join('\\textbf{' + TEX_NAME.get(c, c) + '}' + (UP if higher[c] else DOWN) for c in df.columns)
    lines = ['\\begin{table*}[t]', f'\\caption{{{caption}}}', f'\\label{{{label}}}', '\\centering', '\\scriptsize',
             '\\setlength{\\tabcolsep}{3pt}', '\\begin{tabular}{l' + 'c' * len(df.columns) + '}', '\\toprule',
             f'\\textbf{{{first_col}}} & {hdr} \\\\', '\\midrule']
    for idx in df.index:
        cells = []
        for c in df.columns:
            v = df.loc[idx, c]
            if pd.isna(v):
                cells.append('---')
            else:
                s = fmt.format(float(v)) if 'Time' not in TEX_NAME.get(c, c) else f'{float(v):.1f}'
                cells.append(f'\\textbf{{{s}}}' if best.get(c) == idx else s)
        lines.append(f'{idx} & ' + ' & '.join(cells) + ' \\\\')
    lines += ['\\bottomrule', '\\end{tabular}']
    if foot:
        lines.append(f'\\\\[2pt]\\parbox{{\\textwidth}}{{\\scriptsize {foot}}}')
    lines.append('\\end{table*}')
    return '\n'.join(lines)


tex_xai = latex_table(
    H, HIGHER, 'Extended evaluation of five post-hoc explainability methods on the gated fusion model (TCGA-PAAD, $n=177$). '
    'Best value per column in bold; --- = not applicable.', 'tab:xai_extended',
    foot='Del./Ins. AUC: mean C-index change over $k\\in\\{1,\\dots,100\\}$ minus the random-feature baseline. '
         'Comp./Suff.: risk change in cohort-SD units, mean over $k\\in\\{5,10,20\\}$. Sanity: $|$Spearman $\\rho|$ between '
         'trained- and randomised-model rankings. F1@10/AUPRC: recovery of 10 planted genes, mean of 3 semi-synthetic replicates. '
         'Max-sens.: SHAP, LIME, IG only. PDP is evaluated on its candidate features only.')
P = pred.set_index('model')
PT = pd.DataFrame({'Harrell C': P['harrell_mean'], 'Uno C': P['uno_mean'], 'AUC@12': P['auc12_mean'], 'AUC@24': P['auc24_mean'],
                   'IBS': P['ibs_mean'], 'F1@24': P['F1@24'], 'Bal. acc.@24': P['bal_acc@24'], 'Log-rank p': P['logrank_p_oof']})
PH = {'Harrell C': True, 'Uno C': True, 'AUC@12': True, 'AUC@24': True, 'IBS': False, 'F1@24': True, 'Bal. acc.@24': True, 'Log-rank p': False}
tex_pred = latex_table(PT, PH, 'Out-of-fold predictive performance of unimodal and multimodal survival models (5-fold CV).',
                       'tab:pred_extended', first_col='Model',
                       foot=f'IBS of the Kaplan--Meier reference = {P["ibs_km_mean"].iloc[0]:.3f}. F1 and balanced accuracy '
                            f'for 24-month mortality at the training-fold median risk; patients censored before 24 months excluded. '
                            f'Log-rank p from out-of-fold high/low risk groups.')
with open(f'{EXT_DIR}/table_xai_extended.tex', 'w') as fh:
    fh.write(tex_xai)
with open(f'{EXT_DIR}/table_predictive_extended.tex', 'w') as fh:
    fh.write(tex_pred)
H.to_csv(f'{EXT_DIR}/headline_xai_table.csv'); R.to_csv(f'{EXT_DIR}/rank_table.csv')
with open(f'{EXT_DIR}/extended_results.json', 'w') as fh:
    json.dump(EXT, fh, indent=2, default=lambda o: float(o) if np.isscalar(o) else str(o))
print(f"\nSaved to {EXT_DIR}: headline_xai_table.csv, rank_table.csv, table_xai_extended.tex, "
      f"table_predictive_extended.tex, extended_results.json, figures, per-metric CSVs.")
print("Send back: headline_xai_table.csv, rank_table.csv, B_predictive_metrics.csv and the printed outputs of E2-E9.")

## Preprocessing check (v3)

v3 log-transforms the raw RSEM expression (`log2(x + 1)`) before clipping and scaling, and saves everything to a new Drive folder (`pancreatic_paper_checkpoints_log2`), so nothing from the earlier raw-scale run is loaded by mistake. The raw-scale results stay untouched in the old folder and are compared in E12 as an ablation.

In [ ]:
# ===== E11. Diagnostic: do importances still just follow how extreme a gene's scaled values are? =====
from lifelines import CoxPHFitter
G_sc = X_all[:, CLIN_DIM:]
print(f"Expression after log2, before scaling: median {np.median(X_genomic):.2f}, max {X_genomic.max():.2f}")
spread = np.abs(G_sc - np.median(G_sc, axis=0)).mean(axis=0)
print(f"Scaled spread: median gene {np.median(spread):.2f}, most extreme gene {spread.max():.2f}\n")
_rand = load_checkpoint('extended_eval_sanity')['rand_glob']
diag = pd.DataFrame({'method': METHODS,
                     'rho_importance_vs_spread_trained': [spearmanr(global_importances[m][CLIN_DIM:], spread).correlation for m in METHODS],
                     'rho_importance_vs_spread_random':  [spearmanr(_rand[m][CLIN_DIM:], spread).correlation for m in METHODS],
                     'sanity_top10_overlap': m3['top10_overlap'].values})
print("Low correlations and low sanity overlap = attributions reflect the model, not the input scale.")
print(diag.round(3).to_string(index=False))

top10 = np.argsort(-global_importances['IG'][CLIN_DIM:])[:10]
rows = []
for i in top10:
    d = pd.DataFrame({'x': rankdata(G_sc[:, i]) / len(G_sc), 'T': T, 'E': E})
    cph = CoxPHFitter(penalizer=0.01).fit(d, 'T', 'E')
    rows.append({'gene': gene_cols[i], 'scaled_spread': spread[i],
                 'univariate_HR_low_to_high': cph.hazard_ratios_['x'], 'univariate_p': cph.summary.loc['x', 'p']})
print("\nIG top-10 genes and their model-free survival association (about 1 in 20 would be p < 0.05 by chance):")
print(pd.DataFrame(rows).round(4).to_string(index=False))
EXT['diagnostic'] = {'spread_correlations': diag.set_index('method').to_dict('index'), 'ig_top10_univariate': rows}
diag.to_csv(f'{EXT_DIR}/E11_scale_diagnostic.csv', index=False)

In [ ]:
# ===== E12. Ablation: raw-scale run (old folder) vs log2 run (this folder) =====
# Shows how much the preprocessing choice alone changed the predictions and the explanations.
def _load_from(folder, name):
    p = f'{folder}/{name}.pkl'
    if not os.path.exists(p):
        return None
    with open(p, 'rb') as f:
        return pickle.load(f)

raw_glob = _load_from(RAW_CHECKPOINT_DIR, 'extended_eval_globals')
raw_cv   = _load_from(RAW_CHECKPOINT_DIR, 'cv_state')
if raw_glob is None:
    print(f"No raw-scale results found in {RAW_CHECKPOINT_DIR} -- skipping the ablation.")
else:
    raw_gi = raw_glob['global_importances']
    abl = []
    for m in METHODS:
        mask = np.ones(N_FEAT, bool)
        if m == 'PDP':  # PDP only scores its candidate features in each run
            mask = np.isin(np.arange(N_FEAT), sorted(set(raw_glob['candidates']) | set(cand_list)))
        rg = [i for i in np.argsort(-raw_gi[m]) if i >= CLIN_DIM][:20]
        lg = [i for i in np.argsort(-global_importances[m]) if i >= CLIN_DIM][:20]
        abl.append({'method': m, 'spearman_raw_vs_log2': spearmanr(raw_gi[m][mask], global_importances[m][mask]).correlation,
                    'top20_genes_shared': len(set(rg) & set(lg)),
                    'raw_top5': ', '.join(feature_names[i] for i in rg[:5]),
                    'log2_top5': ', '.join(feature_names[i] for i in lg[:5])})
    abl = pd.DataFrame(abl)
    pd.set_option('display.max_colwidth', 80)
    print(abl.round(3).to_string(index=False))
    if raw_cv is not None:
        print("\nC-index (5-fold CV), raw-scale run -> log2 run:")
        for k in ['clinical', 'genomic', 'fusion']:
            print(f"  {k:9s} {np.mean(raw_cv['results_cv'][k]):.4f} -> {np.mean(results_cv[k]):.4f}")
    EXT['preprocessing_ablation'] = abl.set_index('method').to_dict('index')
    abl.to_csv(f'{EXT_DIR}/E12_preprocessing_ablation.csv', index=False)
with open(f'{EXT_DIR}/extended_results.json', 'w') as fh:
    json.dump(EXT, fh, indent=2, default=lambda o: float(o) if np.isscalar(o) else str(o))
print(f"\nUpdated {EXT_DIR}/extended_results.json")